# 08 — Risk Tools
Test each deterministic risk signal in `RiskTools` against a live Neo4j instance.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
import pandas as pd
from src.config import get_neo4j_settings
from src.storage.neo4j_repository import Neo4jRepository
from src.tools.risk_tools import RiskTools

settings = get_neo4j_settings()
repo = Neo4jRepository(**vars(settings))
repo.test_connection()
risk = RiskTools(repo)
print("Connected")

Connected


In [3]:
def show(r):
    icon = "✓" if r.success else "✗"
    print(f"[{icon}] {r.tool_name}  ({r.duration_ms} ms)")
    print(f"    INPUT  : {r.input}")
    print(f"    SUMMARY: {r.summary}")
    if not r.success:
        print(f"    ERROR  : {r.error}")
    return r

In [4]:
COMPANY = "TELEFONICA O2 HOLDINGS LIMITED"

## ownership_complexity_check

In [5]:
r = show(risk.ownership_complexity_check(COMPANY, max_depth=5))
if r.data:
    signals = {k: v for k, v in r.data.items() if k != "ubos"}
    display(pd.DataFrame([signals]))
    if r.data["ubos"]:
        print("\nIndividual UBOs:")
        display(pd.DataFrame(r.data["ubos"]))

[✓] ownership_complexity_check  (38.4 ms)
    INPUT  : {'company_name': 'TELEFONICA O2 HOLDINGS LIMITED', 'max_depth': 5}
    SUMMARY: Ownership chain for 'TELEFONICA O2 HOLDINGS LIMITED': max depth 1, 1 unique owner(s), 0 individual UBO(s). Complexity risk: MEDIUM. All chains terminate at corporate entities — no individual UBOs resolved.


,max_chain_depth,unique_owner_count,corporate_owner_count,path_count,ubo_count,has_individual_ubos,corporate_chain_only,risk_level
0,1,1,1,1,0,False,True,MEDIUM


## control_signal_check

In [6]:
r = show(risk.control_signal_check(COMPANY, max_depth=5))
if r.data:
    print(f"\nAll control types:")
    for ct in r.data["all_control_types"]:
        flag = " ← ELEVATED" if ct in r.data["elevated_control_types"] else ""
        print(f"  {ct}{flag}")

[✓] control_signal_check  (38.7 ms)
    INPUT  : {'company_name': 'TELEFONICA O2 HOLDINGS LIMITED', 'max_depth': 5}
    SUMMARY: 'TELEFONICA O2 HOLDINGS LIMITED' ownership is via standard share ownership only. Control risk: LOW.

All control types:
  ownership-of-shares-75-to-100-percent


## address_risk_check

In [7]:
r = show(risk.address_risk_check(COMPANY, same_address_threshold=5))
if r.data:
    signals = {k: v for k, v in r.data.items() if k not in ("address",)}
    display(pd.DataFrame([signals]))

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `locality` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=6, column=19, offset=222>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 222, 'line': 6, 'column': 19}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n            MATCH (c:Company {name: $name})-[:REGISTERED_AT]->(a:Address)\n            RETURN\n                a.address_line_1    AS address_line_1,\n                a.address_line_2    AS address_line_2,\n                a.locality          AS locality,\n                a.region            AS region,\n     

[✓] address_risk_check  (46.6 ms)
    INPUT  : {'company_name': 'TELEFONICA O2 HOLDINGS LIMITED', 'same_address_threshold': 5}
    SUMMARY: 'TELEFONICA O2 HOLDINGS LIMITED' shares address (None) with 131 other companies (126 active, 5 dissolved — 4% dissolution rate). Address risk: MEDIUM.


,co_located_total,co_located_active,co_located_dissolved,dissolution_rate,exceeds_threshold,risk_level
0,131,126,5,0.04,True,MEDIUM


## industry_context_check

In [8]:
r = show(risk.industry_context_check(COMPANY))
if r.data:
    print("\nSIC codes:")
    display(pd.DataFrame(r.data["sic_codes"]) if r.data["sic_codes"] else "none")
    if r.data["high_scrutiny_sic_codes"]:
        print("\nHigh-scrutiny codes:")
        display(pd.DataFrame(r.data["high_scrutiny_sic_codes"]))

[✓] industry_context_check  (259.2 ms)
    INPUT  : {'company_name': 'TELEFONICA O2 HOLDINGS LIMITED'}
    SUMMARY: 'TELEFONICA O2 HOLDINGS LIMITED' SIC codes [None, None] are standard. Peer dissolution rate: 7% (13/200). Industry risk: LOW.

SIC codes:


,sic_code,sic_description
0,None,Other telecommunications activities
1,None,Non-trading company


## Combined risk summary

In [9]:
results = [
    risk.ownership_complexity_check(COMPANY),
    risk.control_signal_check(COMPANY),
    risk.address_risk_check(COMPANY),
    risk.industry_context_check(COMPANY),
]

summary_rows = [
    {
        "tool": r.tool_name,
        "risk_level": r.data.get("risk_level", "?") if r.data else "ERROR",
        "ms": r.duration_ms,
        "summary": r.summary,
    }
    for r in results
]

display(pd.DataFrame(summary_rows))

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `locality` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=6, column=19, offset=222>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 222, 'line': 6, 'column': 19}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n            MATCH (c:Company {name: $name})-[:REGISTERED_AT]->(a:Address)\n            RETURN\n                a.address_line_1    AS address_line_1,\n                a.address_line_2    AS address_line_2,\n                a.locality          AS locality,\n                a.region            AS region,\n     

,tool,risk_level,ms,summary
0,ownership_complexity_check,MEDIUM,40.0,Ownership chain for 'TELEFONICA O2 HOLDINGS LI...
1,control_signal_check,LOW,42.0,'TELEFONICA O2 HOLDINGS LIMITED' ownership is ...
2,address_risk_check,MEDIUM,46.1,'TELEFONICA O2 HOLDINGS LIMITED' shares addres...
3,industry_context_check,LOW,243.3,'TELEFONICA O2 HOLDINGS LIMITED' SIC codes [No...


## Close

In [10]:
repo.close()
print("Driver closed")

Driver closed
